# MAADS-BML Holdout Scoring

Scores a holdout CSV through the MAADS-BML PredictionService using:

```python
maadsbml.hyperpredictions(pkey, inputdata, host, port, username)
```
Required `.env` keys:
- `MAADS_HOST`
- `MAADS_PREDICTION_PORT`
- `MAADS_USERNAME`
- `HOLDOUT_CSV`
- `OUTPUT_DIR`
- `ALGO_KEY`

Optional `.env` keys:
- `PRED_THRESHOLD`
- `MAX_ROWS_TO_SCORE`
- `SLEEP_SEC_BETWEEN_CALLS`
- `DOTENV_PATH`


In [1]:
# Imports

import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import nest_asyncio
nest_asyncio.apply()

import maadsbml

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

assert hasattr(maadsbml, 'hyperpredictions'), 'maadsbml.hyperpredictions missing (wrong package/version loaded).'


In [2]:
# Load environment

from dotenv import load_dotenv

dotenv_path = Path(os.environ.get(
    "DOTENV_PATH",
    Path.cwd().parent / ".env"
))
if not dotenv_path.exists():
    raise FileNotFoundError(
        f'.env not found at: {dotenv_path}. ' 
        'Start Jupyter from repo root or set DOTENV_PATH.'
    )

load_dotenv(dotenv_path=dotenv_path, override=False)


True

In [3]:
# Configuration (from .env)

def env_required(key: str) -> str:
    v = os.environ.get(key)
    if v is None or str(v).strip() == '':
        raise KeyError(f'Missing required .env key: {key}')
    return str(v).strip()

host = env_required('MAADS_HOST').rstrip('/')
predictionport = int(env_required('MAADS_PREDICTION_PORT'))
username = env_required('MAADS_USERNAME')

HOLDOUT_CSV = Path(env_required('HOLDOUT_CSV')).expanduser().resolve()
OUTPUT_DIR = Path(env_required('OUTPUT_DIR')).expanduser().resolve()

PRED_THRESHOLD = float(os.environ.get('PRED_THRESHOLD', '0.45') or 0.5)
MAX_ROWS_TO_SCORE = int(os.environ.get('MAX_ROWS_TO_SCORE', '0') or 0)
SLEEP_SEC_BETWEEN_CALLS = float(os.environ.get('SLEEP_SEC_BETWEEN_CALLS', '0') or 0)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# Resolve AlgoKey from MAADS exception folder

import os

def env_required(key: str) -> str:
    v = os.environ.get(key)
    if v is None or str(v).strip() == "":
        raise KeyError(f"Missing required .env key: {key}")
    return str(v).strip()

def latest_algokey_from_exception(exception_dir: Path) -> str:
    if not exception_dir.exists():
        raise FileNotFoundError(f"Missing exception directory: {exception_dir}")

    candidates = sorted(
        exception_dir.glob("*trained_algo*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No '*trained_algo*.json' files found in: {exception_dir}"
        )

    for p in candidates:
        try:
            meta = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
        except Exception:
            continue

        if isinstance(meta, dict):
            algokey = str(meta.get("AlgoKey") or "").strip()
            if algokey:
                return algokey

    raise RuntimeError(
        f"Found trained_algo JSONs but none contained 'AlgoKey' in: {exception_dir}"
    )

# Resolve MAADS_ROOT from environment
MAADS_ROOT = Path(env_required("MAADS_ROOT")).expanduser().resolve()
exception_dir = MAADS_ROOT / "exception"
pkey = latest_algokey_from_exception(exception_dir)

print("Resolved AlgoKey:", pkey)


Resolved AlgoKey: admin_maads_train_aug_binary_csv


In [5]:
# Load holdout

if not HOLDOUT_CSV.exists():
    raise FileNotFoundError(f"Holdout CSV not found: {HOLDOUT_CSV}")

df = pd.read_csv(HOLDOUT_CSV)
print("Total rows:", len(df))

n = len(df) if MAX_ROWS_TO_SCORE <= 0 else min(len(df), MAX_ROWS_TO_SCORE)
df = df.iloc[:n].copy()

df.head()


Total rows: 754


,Date,customerage,transactionduration,loginattempts,txn_hour,is_night,is_weekend,mcc,high_risk_mcc,medium_risk_mcc,new_device,new_ip,customer_tenure_days,short_tenure,unusual_mcc_for_customer,txn_velocity_5min_sim,high_frequency,freq_2plus,proxy,transactiontype_Credit,transactiontype_Debit,transactiontype_nan,channel_Branch,channel_Online,channel_nan,customeroccupation_Doctor,customeroccupation_Engineer,customeroccupation_Retired,customeroccupation_Student,customeroccupation_nan,mcc_risk_low,mcc_risk_medium,mcc_risk_nan,country_BR,country_CA,country_DE,country_FR,country_GB,country_IN,country_MX,country_PL,country_SG,country_UA,country_US,country_nan,label
0,10/06/2023,24.0,150.0,1.0,16.0,0.0,0.0,5921.0,1.0,0.0,1.0,1.0,698.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
1,05/16/2023,42.0,74.0,1.0,16.0,0.0,0.0,5311.0,0.0,0.0,1.0,1.0,1024.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
2,01/27/2023,61.0,125.0,1.0,18.0,0.0,0.0,5732.0,0.0,1.0,1.0,1.0,176.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
3,09/19/2023,22.0,123.0,1.0,17.0,0.0,0.0,4121.0,0.0,0.0,1.0,1.0,115.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,10/19/2023,21.0,135.0,1.0,16.0,0.0,0.0,4111.0,0.0,0.0,0.0,1.0,69.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0


In [6]:
# Resolve FEATURE_COLUMNS from MAADS artifacts

def _parse_maads_json(raw):
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        s = raw.strip()
        if s.startswith("{") and s.endswith("}"):
            return json.loads(s)
    return {"error": "Unexpected MAADS response type", "raw_type": str(type(raw)), "raw": raw}

def _bootstrap_cols_from_prediction_details(maads_root: Path, algo_key: str) -> list[str]:
    csv_path = maads_root / "csvuploads" / f"{algo_key}_prediction_details.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"prediction_details.csv not found: {csv_path}")

    details = pd.read_csv(csv_path)

    drop = {c for c in details.columns if c.strip().startswith("[")}
    drop |= {"label"}  # ignore label if present

    cols = [c for c in details.columns if c not in drop]
    if not cols:
        raise RuntimeError(f"No feature columns detected in: {csv_path}")

    print("Bootstrap feature source:", csv_path)
    print("Bootstrap feature count:", len(cols))
    return cols

def _row_to_inputdata_with_cols(row: pd.Series, cols: list[str]) -> str:
    out = []
    for c in cols:
        v = row[c]
        if pd.isna(v):
            out.append("")
            continue
        s = str(v)
        # If a value contains commas, quote it to preserve column alignment.
        if "," in s:
            s = '"' + s.replace('"', '""') + '"'
        out.append(s)
    return ",".join(out)

def _fields_to_cols(fields_str: str) -> list[str]:
    cols = [c.strip() for c in (fields_str or "").split(",") if c.strip()]
    if not cols:
        raise RuntimeError("MAADS response 'Fields' is empty; cannot resolve schema.")
    return cols

# If FEATURE_COLUMNS=AUTO in .env, we resolve automatically.
FEATURE_COLUMNS_ENV = (os.environ.get("FEATURE_COLUMNS", "AUTO") or "AUTO").strip()

if FEATURE_COLUMNS_ENV.upper() != "AUTO":
    FEATURE_COLUMNS = [c.strip() for c in FEATURE_COLUMNS_ENV.split(",") if c.strip()]
    print("Feature columns source: .env FEATURE_COLUMNS (explicit)")
    print("Feature columns count:", len(FEATURE_COLUMNS))

else:
    MAADS_ROOT_PATH = Path(env_required("MAADS_ROOT")).expanduser().resolve()

    # 1) Bootstrap order from prediction_details.csv
    bootstrap_cols = _bootstrap_cols_from_prediction_details(MAADS_ROOT_PATH, pkey)

    # Ensure holdout contains everything required by bootstrap order.
    missing = [c for c in bootstrap_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Holdout CSV missing columns required by prediction_details: {missing}")

    # 2) Call to PredictionService to fetch authoritative "Fields"
    sample_input = _row_to_inputdata_with_cols(df.iloc[0], bootstrap_cols)
    raw = maadsbml.hyperpredictions(pkey, sample_input, host, predictionport, username)
    resp = _parse_maads_json(raw)

    fields_cols = _fields_to_cols(resp.get("Fields", ""))

    print("Authoritative Fields count:", len(fields_cols))

    # 3) Fail on any mismatch
    if bootstrap_cols != fields_cols:
        for i, (a, b) in enumerate(zip(bootstrap_cols, fields_cols)):
            if a != b:
                raise RuntimeError(
                    f"Feature order mismatch at position {i}: prediction_details='{a}' vs Fields='{b}'. "
                )
        raise RuntimeError("Feature order mismatch detected (length/order differs).")

    FEATURE_COLUMNS = fields_cols
    print("FEATURE_COLUMNS resolved from MAADS Fields and validated.")

print("First 10 FEATURE_COLUMNS:", FEATURE_COLUMNS[:10])
print("Feature columns:", len(FEATURE_COLUMNS))


Bootstrap feature source: /home/user/projects/tml-fraud/supervised/maads_service/csvuploads/admin_maads_train_aug_binary_csv_prediction_details.csv
Bootstrap feature count: 45
Authoritative Fields count: 45
FEATURE_COLUMNS resolved from MAADS Fields and validated.
First 10 FEATURE_COLUMNS: ['Date', 'customerage', 'transactionduration', 'loginattempts', 'txn_hour', 'is_night', 'is_weekend', 'mcc', 'high_risk_mcc', 'medium_risk_mcc']
Feature columns: 45


In [8]:
# Row serialization

def row_to_inputdata(row: pd.Series) -> str:
    out = []
    for col in FEATURE_COLUMNS:
        v = row[col]
        if pd.isna(v):
            out.append('')
            continue
        s = str(v)
        if ',' in s:
            s = '"' + s.replace('"', '""') + '"'
        out.append(s)
    return ','.join(out)

sample = row_to_inputdata(df.iloc[0])
print('Serialized fields:', sample.count(',') + 1)
print('Preview:', sample[:200] + ('...' if len(sample) > 200 else ''))


Serialized fields: 45
Preview: 10/06/2023,24.0,150.0,1.0,16.0,0.0,0.0,5921.0,1.0,0.0,1.0,1.0,698.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [9]:
# Response normalization

def normalize_maads_response(r):
    if isinstance(r, dict):
        return r
    if isinstance(r, str):
        s = r.strip()
        if s.startswith("{") and s.endswith("}"):
            return json.loads(s)
    return {"error": "Unexpected response type", "raw_type": str(type(r)), "raw": r}


In [10]:
# Scoring

responses_raw = []
p0 = []
p1 = []
pred = []
failures = 0

t0 = time.time()
print('Scoring rows:', len(df))

for i in range(len(df)):
    inputdata = row_to_inputdata(df.iloc[i])
    try:
        raw = maadsbml.hyperpredictions(pkey, inputdata, host, predictionport, username)
        r = normalize_maads_response(raw)
    except Exception as e:
        r = {'error': str(e)}

    responses_raw.append(r)

    hp = r.get('hyperprediction') if isinstance(r, dict) else None
    if isinstance(hp, list) and len(hp) == 2 and hp[0] != -1 and hp[1] != -1:
        _p0 = float(hp[0])
        _p1 = float(hp[1])
        p0.append(_p0)
        p1.append(_p1)
        pred.append(int(_p1 >= PRED_THRESHOLD))
    else:
        failures += 1
        p0.append(np.nan)
        p1.append(np.nan)
        pred.append(None)

    if SLEEP_SEC_BETWEEN_CALLS > 0:
        time.sleep(SLEEP_SEC_BETWEEN_CALLS)

    if (i + 1) % 100 == 0 or (i + 1) == len(df):
        print(f'{i+1}/{len(df)} done | failures={failures} | elapsed={time.time()-t0:.1f}s')

print('Failures:', failures)


Scoring rows: 754
100/754 done | failures=0 | elapsed=1.4s
200/754 done | failures=1 | elapsed=3.0s
300/754 done | failures=1 | elapsed=4.5s
400/754 done | failures=1 | elapsed=6.1s
500/754 done | failures=1 | elapsed=7.6s
600/754 done | failures=1 | elapsed=9.0s
700/754 done | failures=1 | elapsed=10.4s
754/754 done | failures=1 | elapsed=11.2s
Failures: 1


In [11]:
# Persist artifacts

scored_df = df.copy()
scored_df['maads_p0'] = p0
scored_df['maads_p1_fraud'] = p1
scored_df['maads_pred_label'] = pred

out_csv = OUTPUT_DIR / 'holdout_scored.csv'
out_json = OUTPUT_DIR / 'holdout_maads_responses.json'

scored_df.to_csv(out_csv, index=False)

out_json.write_text(
    json.dumps(
        {
            'algo_key': pkey,
            'host': host,
            'prediction_port': predictionport,
            'username': username,
            'scored_rows': int(len(scored_df)),
            'failures': int(failures),
            'threshold': float(PRED_THRESHOLD),
            'feature_columns': FEATURE_COLUMNS,
            'responses': responses_raw,
        },
        indent=2,
    ),
    encoding='utf-8',
)

print('Saved CSV and JSON.')
scored_df.head()


Saved CSV and JSON.


,Date,customerage,transactionduration,loginattempts,txn_hour,is_night,is_weekend,mcc,high_risk_mcc,medium_risk_mcc,new_device,new_ip,customer_tenure_days,short_tenure,unusual_mcc_for_customer,txn_velocity_5min_sim,high_frequency,freq_2plus,proxy,transactiontype_Credit,transactiontype_Debit,transactiontype_nan,channel_Branch,channel_Online,channel_nan,customeroccupation_Doctor,customeroccupation_Engineer,customeroccupation_Retired,customeroccupation_Student,customeroccupation_nan,mcc_risk_low,mcc_risk_medium,mcc_risk_nan,country_BR,country_CA,country_DE,country_FR,country_GB,country_IN,country_MX,country_PL,country_SG,country_UA,country_US,country_nan,label,maads_p0,maads_p1_fraud,maads_pred_label
0,10/06/2023,24.0,150.0,1.0,16.0,0.0,0.0,5921.0,1.0,0.0,1.0,1.0,698.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,0.042,0.958,1.0
1,05/16/2023,42.0,74.0,1.0,16.0,0.0,0.0,5311.0,0.0,0.0,1.0,1.0,1024.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0.756,0.244,0.0
2,01/27/2023,61.0,125.0,1.0,18.0,0.0,0.0,5732.0,0.0,1.0,1.0,1.0,176.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0.918,0.082,0.0
3,09/19/2023,22.0,123.0,1.0,17.0,0.0,0.0,4121.0,0.0,0.0,1.0,1.0,115.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0.686,0.314,0.0
4,10/19/2023,21.0,135.0,1.0,16.0,0.0,0.0,4111.0,0.0,0.0,0.0,1.0,69.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,0.999,0.001,0.0


In [12]:
# Summary

print('p1 (fraud prob) stats:')
print(pd.Series(p1).describe())

if 'label' in scored_df.columns:
    from sklearn.metrics import classification_report, confusion_matrix
    y_true = scored_df['label'].astype(int).values
    y_pred = pd.Series(scored_df['maads_pred_label']).fillna(0).astype(int).values
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))
    print('\nClassification report:')
    print(classification_report(y_true, y_pred, digits=4))
else:
    print('No label column in holdout; skipping classification metrics.')


p1 (fraud prob) stats:
count    753.000000
mean       0.345772
std        0.348430
min        0.001000
25%        0.018000
50%        0.268000
75%        0.599000
max        0.999000
dtype: float64
Confusion matrix:
[[511  78]
 [ 31 134]]

Classification report:
              precision    recall  f1-score   support

           0     0.9428    0.8676    0.9036       589
           1     0.6321    0.8121    0.7109       165

    accuracy                         0.8554       754
   macro avg     0.7874    0.8398    0.8073       754
weighted avg     0.8748    0.8554    0.8614       754

